###Task 3: Model Development 

In [201]:
#Load cleaned dataset
import pandas as pd

model_df = pd.read_csv("cleaned_dataset.csv")

In [202]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25129 entries, 0 to 25128
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              25129 non-null  int64  
 1   gender          25129 non-null  str    
 2   car             25129 non-null  str    
 3   reality         25129 non-null  str    
 4   no_of_child     25129 non-null  int64  
 5   family_type     25129 non-null  str    
 6   house_type      25129 non-null  str    
 7   flag_mobil      25129 non-null  int64  
 8   work_phone      25129 non-null  int64  
 9   phone           25129 non-null  int64  
 10  e_mail          25129 non-null  int64  
 11  family_size     25129 non-null  float64
 12  begin_month     25129 non-null  int64  
 13  age             25129 non-null  int64  
 14  years_employed  25129 non-null  float64
 15  target          25129 non-null  int64  
 16  income          25129 non-null  float64
 17  income_type     25129 non-null  str    
 1

In [203]:
#data encoding
# -------------------------
# -------------------------
# LABEL ENCODING
# -------------------------
from sklearn.preprocessing import LabelEncoder

# Create encoder
#le = LabelEncoder()

# List of categorical columns
categorical_cols = [
    'gender',
    'car',
    'reality',
    'income_type',
    'education_type',
    'family_type',
    'house_type'
]

# Apply encoding
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col])
    encoders[col] = le



In [204]:
#Feature Selection (X and y)
X = model_df.drop('target', axis=1)
y = model_df['target']

In [205]:
#Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [206]:
#Feature Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

###Model Comparison table creation

In [207]:
#Create result storage
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = []

def evaluate_model(name, y_test, y_pred):
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0)
    })

In [208]:
#Logistic Regression - use SCALED data
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, class_weight='balanced')

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

evaluate_model("Logistic Regression (Balanced)", y_test, y_pred_lr)

In [209]:
#Random Forest (scaling)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
#rf.fit(X_train, y_train)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

evaluate_model("Random Forest", y_test, y_pred_rf)

In [210]:
#Decision Tree (NO scaling)
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

evaluate_model("Decision Tree", y_test, y_pred_dt)

In [211]:
#ANN (use SCALED data)
from sklearn.neural_network import MLPClassifier

ann = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=42)

ann.fit(X_train_scaled, y_train)

y_pred_ann = ann.predict(X_test_scaled)

evaluate_model("ANN (MLP)", y_test, y_pred_ann)

In [212]:
results_df = pd.DataFrame(results)
results_df.sort_values(by='Recall', ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression (Balanced),0.639276,0.030601,0.589474,0.058182
2,Decision Tree,0.967171,0.156863,0.168421,0.162437
3,ANN (MLP),0.977318,0.297872,0.147368,0.197183
1,Random Forest,0.979904,0.384615,0.105263,0.165289


##Improve Logistic regression


In [213]:
#Hyperparameter Tuning (Improved Logistic Regression)
from sklearn.linear_model import LogisticRegression

lr_improved = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=0.5,
    solver='liblinear'
)

lr_improved.fit(X_train_scaled, y_train)

y_pred_lr_imp = lr_improved.predict(X_test_scaled)

evaluate_model("Improved Logistic Regression", y_test, y_pred_lr_imp)

In [214]:
#Threshold Adjustment
# Get probability scores
y_prob = lr_improved.predict_proba(X_test_scaled)[:, 1]

# Apply custom threshold
y_pred_custom = (y_prob > 0.01).astype(int)

evaluate_model("LR with Threshold 0.01", y_test, y_pred_custom)

##Imporve Random forest

In [215]:
rf_balanced = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf_balanced.fit(X_train, y_train)

y_pred_rf_bal = rf_balanced.predict(X_test)

evaluate_model("Random Forest (Balanced)", y_test, y_pred_rf_bal)

In [216]:
rf_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
)

rf_tuned.fit(X_train, y_train)

y_pred_rf_tuned = rf_tuned.predict(X_test)

evaluate_model("RF (Tuned + Balanced)", y_test, y_pred_rf_tuned)



In [217]:
y_prob = rf_tuned.predict_proba(X_test)[:, 1]

from sklearn.metrics import roc_curve
import numpy as np

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

j_scores = tpr - fpr

best_index = np.argmax(j_scores)
best_threshold = thresholds[best_index]

print("Best Threshold:", best_threshold)

Best Threshold: 0.35027114271641935


In [218]:
#improved result
results_df = pd.DataFrame(results)
results_df.sort_values(by='Recall', ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score
5,LR with Threshold 0.01,0.018902,0.018902,1.000000,0.037102
0,Logistic Regression (Balanced),0.639276,0.030601,0.589474,0.058182
4,Improved Logistic Regression,0.639276,0.030601,0.589474,0.058182
7,RF (Tuned + Balanced),0.978910,0.396226,0.221053,0.283784
2,Decision Tree,0.967171,0.156863,0.168421,0.162437
3,ANN (MLP),0.977318,0.297872,0.147368,0.197183
1,Random Forest,0.979904,0.384615,0.105263,0.165289
6,Random Forest (Balanced),0.979904,0.350000,0.073684,0.121739


In [219]:
#save model
import joblib

# Save models
joblib.dump(lr_improved, "lr_model.pkl")
#joblib.dump(rf, "rf_model.pkl") #rf_tuned
joblib.dump(rf_tuned, "rf_model.pkl")
# Save scaler
joblib.dump(scaler, "scaler.pkl")
# SAve encodert
joblib.dump(encoders, "encoders.pkl")
# save threshold
joblib.dump(best_threshold, "threshold.pkl")

['threshold.pkl']